# --- STEP 1: Environment Setup and Imports ---

In [ ]:
!pip install lightgbm boruta scikit-learn shap pandas numpy matplotlib statsmodels seaborn deap pymoo imbalanced-learn tabulate torch

In [ ]:
import sys
sys.path.append('../')
import utils

# Python version used
print(sys.version)

# Check the attributes and methods of the `utils` package.
print(dir(utils))

import os

# List files in the parent directory
# Check if the parent directory exists and add it to sys.path
parent_dir = os.path.abspath('../')
if os.path.isdir(parent_dir):
    sys.path.append(parent_dir)
    print("Files in the parent directory:")
    print(os.listdir(parent_dir))
    try:
        import utils
        print("Module 'utils' imported successfully.")
    except ModuleNotFoundError:
        print("The module 'utils' was not found. Verify the path and file existence.")
else:
    print("The parent directory does not exist.")

In [ ]:
pip list

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm # For statistical models, although not directly used in SHAP here
import seaborn as sns
import sklearn
import random
import scipy

import lightgbm as lgb

import shap
# import deap
import pymoo
import time

from imblearn.over_sampling import SMOTE

from lightgbm import LGBMClassifier

from sklearn.base import is_classifier # To check the model type
from sklearn.base import clone
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import Lasso, LogisticRegression, LinearRegression # For Kernel SHAP regression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, roc_curve, precision_recall_curve, f1_score, auc, accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.tree import DecisionTreeClassifier

# from deap import base, creator, tools, algorithms # For genetic algorithms
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.operators.crossover.pntx import TwoPointCrossover
from pymoo.operators.crossover.ux import UniformCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling

from boruta import BorutaPy

import itertools
from math import factorial # For calculating factorials
from scipy.special import comb # For binomial coefficient (M choose k)

from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

print("All Ok imports successful!")

# Initializes the JavaScript environment for viewing interactive SHAP charts
shap.initjs()

print("--- STEP 1: Environment Setup and Imports ---")

# --- STEP 2: Data Loading and Preprocessing ---

In [ ]:
# 1. Load the dataset
# Discover the current folder
current_directory = Path.cwd()

# Load the file
file_path = current_directory / "Datasets" / "Company_Bankruptcy_Prediction" / "data.csv"
df = pd.read_csv(file_path)

# The target is the first column ('Bankrupt?')
# Note: Some versions of CSV include a space in the column names; we use `strip()` to ensure this
df.columns = df.columns.str.strip()

X = df.drop(columns=['Bankrupt?'])
y = df['Bankrupt?']

# 2. Data Cleansing
# Let's remove the 'Net Income Flag' column because it has zero variance (only values ​​of 1)
if 'Net Income Flag' in X.columns:
    X = X.drop(columns=['Net Income Flag'])

# 3. Division into Training and Testing
# The stratify=y setting ensures that the proportion of failures is the same in training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Feature Scheduling
# RobustScaler is ideal here because it handles outliers in financial indicators better
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame to retain column names (optional, but recommended)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# 5. Class Balancing (Applied only to training data)
# SMOTE creates synthetic samples of the minority class (companies that went bankrupt)
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

In [ ]:
# Viewing the first few lines of the dataset
df.shape
df.info()
df.head(10)

In [ ]:
print(df.describe().T)

In [ ]:
print(df.isnull().sum().sum(), "missing values found")

# --- STEP 3: Multi-Objective Formulation (NSGA-II via Pymoo) ---

In [ ]:
# ==========================================
# 1. Mathematical Formulation (Fitness Function)
# ==========================================
class FeatureSelectionProblem(ElementwiseProblem):
    def __init__(self, X_train, y_train, **kwargs):
        self.n_features = X_train.shape[1]
        # n_var = total de features (bits)
        # n_obj = 2 objectives (Gini and Number of Features)
        super().__init__(n_var=self.n_features, 
                         n_obj=2, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        
        # Converting to NumPy arrays if they are pandas DataFrames
        self.X_train = np.array(X_train)
        self.y_train = np.array(y_train)

    def _evaluate(self, x, out, *args, **kwargs):
        # x is a boolean array (Ex: [True, False, True...])
        selected_features = np.where(x)[0]
        
        # Severe punishment if the algorithm resets all features
        if len(selected_features) == 0:
            out["F"] = [1.0, self.n_features]
            return

        # Limiting the depth of the tree to avoid internal overfitting
        # Training a fast model to evaluate the subset
        clf = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
        X_subset = self.X_train[:, selected_features]
        
        # REPEATED K-FOLD
        # 5 Folds, repeated 3 times (total of 15 training sessions per individual)
        cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
        
        # The cross_val_score trains and evaluates the internal evaluation metric for ROC AUC
        scores_auc = cross_val_score(clf, X_subset, self.y_train, cv=cv, scoring='roc_auc')

        # We calculated the average AUC of the validations
        auc_average = scores_auc.mean()

        # We applied the Gini mathematical formula
        gini_average = (2 * auc_average) - 1
        
        # Objective 1: Minimize (1 - Gini_Average)
        f1 = 1.0 - gini_average
        
        # Objective 2: Minimize the number of features
        f2 = len(selected_features)
        
        out["F"] = [f1, f2]

# ==========================================
# 2. NSGA-II CONFIGURATION AND EXECUTION
# ==========================================
# Instantiate the problem using the scaled/balanced data from Step 2
# Note: Assuming we split X_train into training and validation internally
problem = FeatureSelectionProblem(X_train_balanced, y_train_balanced)

# Configure the Genetic Algorithm
algorithm = NSGA2(
    pop_size=100,
    sampling=BinaryRandomSampling(),
    crossover=UniformCrossover(),
    mutation=BitflipMutation(),
    eliminate_duplicates=True
)

# Run the optimization
print("Starting NSGA-II optimization...")
start_time_nsga = time.time()
res = minimize(problem,
               algorithm,
               ('n_gen', 100), # Number of generations
               seed=42,
               verbose=True)
execution_time_nsga2 = time.time() - start_time_nsga
minutes = int(execution_time_nsga2 // 60)
seconds = execution_time_nsga2 % 60
print(f"Optimization completed in {minutes}m {seconds:.2f}s, mathematical total: {execution_time_nsga2:.2f} seconds)")

# ==========================================
# 3. EXTRACTION OF THE IDEAL MODEL (PARETO FRONTIER)
# ==========================================
# res.F contains the fitness values ​​(Gini Score, Number of Features)
# res.X contains the chromosomes (boolean arrays)

# Choosing the model based on the "Sweet Point" (The one that balances high Gini Score and few features)
# We use the Pseudo-Weights technique to find the best balance at the boundary
from pymoo.mcdm.pseudo_weights import PseudoWeights

# Weighting: 70% importance for the gini metric, 30% for feature reduction.
weights = np.array([0.7, 0.3]) 
best_idx = PseudoWeights(weights).do(res.F)

better_chromosome = res.X[best_idx]
better_gini = 1 - res.F[best_idx][0]
qtd_features = res.F[best_idx][1]

final_features_names = X_train_balanced.columns[better_chromosome].tolist()

print("\n--- IDEAL RESULT EXTRACTED ---")
print(f"Gini achieved: {better_gini:.4f}")
print(f"Number of features used: {int(qtd_features)}")
print(f"Selected features:\n {list(final_features_names)}")

In [ ]:
# ==========================================
# 4. VISUALIZATION OF THE PARETO BOUNDARY
# ==========================================

# 1. Extracting the values ​​of all models from the Frontier (res.F)
# Objective 1 was (1 - Gini), so we converted it back to Gini by multiplying by -1 and adding 1
gini_scores_border = 1.0 - res.F[:, 0]

# Objective 2 is already the exact number of features
qtd_features_border = res.F[:, 1]

# 2. Setting up the chart figure
plt.figure(figsize=(10, 6))

# 3. Plotting all the models found on the Border (in blue)
plt.scatter(qtd_features_border, gini_scores_border, 
            color='royalblue', s=80, alpha=0.8, edgecolors='black', 
            label='Optimal Models (Pareto Frontier)')

# 4. Highlighting the "Champion Model" chosen by weights 70/30 (in red)
# Note: 'qtd_features' and 'better_gini' are the variables calculated at the end of Step 3
plt.scatter(qtd_features, better_gini, 
            color='red', marker='*', s=300, edgecolors='black', 
            label='Selected Model (Weights 70/30)')

# 5. Adding the Gini values ​​above each point to make it easier to read
for i in range(len(qtd_features_border)):
    plt.annotate(f"{gini_scores_border[i]:.3f}", 
                 (qtd_features_border[i], gini_scores_border[i]),
                 textcoords="offset points", 
                 xytext=(0,10), # Text displacement so it doesn't overlap the period
                 ha='center',
                 fontsize=9)

# 6. Styling the chart (Titles, Axes, and Grid)
plt.title('Trade-off between Performance and Interpretability (NSGA-II)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Model Complexity (Number of Features Selected)', fontsize=12)
plt.ylabel('Gini Score', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', frameon=True, fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
dictionary_features = {}
execution_time = {}

In [ ]:
# Dictionary that will store the column lists for each method
dictionary_features['NSGA-II'] = final_features_names
execution_time['NSGA-II'] = execution_time_nsga2
print(f"NSGA-II already processed: {len(dictionary_features['NSGA-II'])} features em {execution_time_nsga2:.2f}s")

features_nsga2 = dictionary_features['NSGA-II']
print(f"=== {len(features_nsga2)} features selected by NSGA-II ===")
for i, feature in enumerate(features_nsga2, 1):
    print(f"{i}. {feature}")

In [ ]:
print("\n=== EXPLORING THE PARETO BOUNDARY ===")

# We checked if NSGA-II found solutions (res.F contains the objectives and res.X the variables)
if res.X is not None:
    # If there is only one optimal solution, res.X and res.F will not be lists of lists,
    # therefore we ensure that they are iterable:
    X_finals = res.X if res.X.ndim > 1 else [res.X]
    F_finals = res.F if res.F.ndim > 1 else [res.F]

    for i, (objectives, chromosome) in enumerate(zip(F_finals, X_finals)):
          
        gini_real = 1 - abs(objectives[0])
        qtd_variaveis = int(sum(chromosome))
        
        print(f"Solution {i+1:2d}: {int(qtd_variaveis):2d} variables | Gini real: {gini_real:.4f}")
else:
    print("No solution found.")

## Competition

In [ ]:
# ==========================================
# BORUTA (Using its best feature: Random Forest)
# ==========================================

start_time = time.time()

rf_boruta = RandomForestClassifier(n_jobs=-1, class_weight='balanced', max_depth=5, random_state=42)
boruta_selector = BorutaPy(rf_boruta, n_estimators='auto', verbose=0, random_state=42, max_iter=50)
boruta_selector.fit(X_train_balanced.values, y_train_balanced.values)
dictionary_features['Boruta'] = X_train_balanced.columns[boruta_selector.support_].tolist()

execution_time['Boruta'] = time.time() - start_time

print(f"Boruta selected {len(dictionary_features['Boruta'])} features in {execution_time['Boruta']:.2f}s")

In [ ]:
# ==========================================
# LASSO
# ==========================================

start_time = time.time()

lasso = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, random_state=42, class_weight='balanced')
lasso_selector = SelectFromModel(lasso)
lasso_selector.fit(X_train_balanced, y_train_balanced)

dictionary_features['Lasso'] = X_train_balanced.columns[lasso_selector.get_support()].tolist()
execution_time['Lasso'] = time.time() - start_time

print(f"Lasso selected {len(dictionary_features['Lasso'])} features in {execution_time['Lasso']:.2f}s")

In [ ]:
# ==========================================
# RFE
# ==========================================
# Dynamically forced to find the same number of features as NSGA-II
qtd_features_of_nsga2 = len(dictionary_features['NSGA-II']) 
print(f"\nRFE (Configured to search {qtd_features_of_nsga2} features)...")
start_time = time.time()

rfe = RFE(estimator=DecisionTreeClassifier(random_state=42, max_depth=5), n_features_to_select=qtd_features_of_nsga2, step=1)
rfe.fit(X_train_balanced, y_train_balanced)

dictionary_features['RFE'] = X_train_balanced.columns[rfe.support_].tolist()
execution_time['RFE'] = time.time() - start_time

print(f"RFE selected {len(dictionary_features['RFE'])} features in {execution_time['RFE']:.2f}s")

In [ ]:
features_nsga = set(dictionary_features['NSGA-II'])
features_rfe = set(dictionary_features['RFE'])

in_common = features_nsga.intersection(features_rfe)

print(f"Common features: {len(in_common)} of {qtd_features_of_nsga2}")
print(f"What are they?: {in_common}")

"Given that Lasso is linear, Boruta uses forests, RFE is greedy, and NSGA-II uses genetic heuristics, which provides the best dataset for a production LightGBM to achieve the highest Gini coefficient?"

# --- STEP 4: The Judgment (in Test Set) ---

In [ ]:
warnings.filterwarnings('ignore') # Hide convergence alerts

print("- STARTING THE BASELINES COMPETITION -")

judgment = {
    'LightGBM': LGBMClassifier(random_state=42, class_weight='balanced', n_estimators=100, n_jobs=-1, verbose=-1),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced', n_estimators=100, n_jobs=-1),
    'Logistic Regression': LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000, solver='liblinear'),
    'MLP': MLPClassifier(random_state=42, max_iter=500, hidden_layer_sizes=(100, 50))
}

final_results = []

# 2. Feature Selectors vs. Classifier Judges
for method_name, column_list in dictionary_features.items():
    
    # We filtered the data only for the columns selected by this method
    X_train_cut = X_train_balanced[column_list]
    X_test_cut = X_test_scaled[column_list] 
    
    for judge_name, judge_model in judgment.items():
        
        # The judge trains...
        judge_model.fit(X_train_cut, y_train_balanced)
        
        # The judge tests...
        y_proba = judge_model.predict_proba(X_test_cut)[:, 1]
        auc_teste = roc_auc_score(y_test, y_proba)
        
        # Gini Calculation
        gini_real_test = (2 * auc_teste) - 1
        
        # Save the result
        final_results.append({
            'Feature Selector': method_name,
            'Classifier (Judge)': judge_name,
            'Selected Features': len(column_list),
            'Gini (Real Test)': gini_real_test,
            'Search Time (s)': execution_time[method_name] 
        })

# 3. Assembly and presentation of the ranking
df_results = pd.DataFrame(final_results)

# Business parameters
gini_weight = 0.70
parsimony_weight = 0.30

# Dynamic capture of the total number of original columns
total_original_features = X_train_balanced.shape[1]

# Creating the Parsimony score (0 to 1, where 1 is the best/most economical)
df_results['Parsimony Score'] = 1 - (df_results['Selected Features'] / total_original_features)

# Calculating the Weighted Final Score
df_results['Final Score'] = (df_results['Gini (Real Test)'] * gini_weight) + (df_results['Parsimony Score'] * parsimony_weight)

df_results = df_results.sort_values(by='Final Score', ascending=False).reset_index(drop=True)

top_method = df_results.iloc[0]['Feature Selector']
top_model = df_results.iloc[0]['Classifier (Judge)']
top_score = df_results.iloc[0]['Final Score']
top_gini = df_results.iloc[0]['Gini (Real Test)']
selected_features_top = df_results.iloc[0]['Selected Features']

# ==============================================================================
df_results_print = df_results.copy()
df_results_print['Gini (Real Test)'] = df_results_print['Gini (Real Test)'].map('{:.3f}'.format)
df_results_print['Parsimony Score'] = df_results_print['Parsimony Score'].map('{:.3f}'.format)
df_results_print['Final Score'] = df_results_print['Final Score'].map('{:.3f}'.format)
if 'Search Time (s)' in df_results_print.columns:
    df_results_print['Search Time (s)'] = df_results_print['Search Time (s)'].map('{:.3f}'.format)

# ==============================================================================
display_columns = ['Feature Selector', 'Classifier (Judge)', 'Selected Features', 'Parsimony Score', 'Gini (Real Test)', 'Final Score']
df_results_print = df_results_print[display_columns]

print("\n=== DEFINITIVE RANKING (MULTIOBJECTIVE SCORE) IN THE REAL SCENARIO ===")
print(df_results_print.to_markdown(index=False))

print("\n" + "="*80)
print(" 📊 MULTIOBJECTIVE TRADE-OFF ANALYSIS")
print("="*80)
print(f"Top Performer : {top_method} running on {top_model}")
print(f"Final Score   : {top_score:.3f} (Gini: {top_gini:.3f} | Features: {selected_features_top}/{total_original_features})")
print("-" * 80)
print("Methodological Note: The Paradox of the Architect vs. The Operator")
print(f"Although methods like {top_method} may achieve top scores efficiently, it is")
print(f"crucial to note they often operate under a heuristic constraint (e.g., n={selected_features_top})")
print("which was previously discovered by the global Pareto search of the NSGA-II.")
print("Therefore, NSGA-II validates the topology (The Architect), while the subsequent")
print("algorithms provide high-speed execution (The Operator) in a synergistic ecosystem.")
print("="*80)

# --- STEP 5: Post-Hoc Validation ---

In [ ]:
# ==========================================
# STEP 5: EXPLAINABLE AI (XAI) WITH SHAP
# ==========================================
print("--- STEP 5: EXTRACTING EXPLAINABILITY FOR THE NSGA-II WINNER ---")

# 1. Filter the results to find the best judge EXCLUSIVELY for the NSGA-II
nsga_results = df_results[df_results['Feature Selector'] == 'NSGA-II']

# Since df_results has already been sorted from best to worst previously, 
# the iloc[0] of this subset will be guaranteed to be the best NSGA-II match
best_nsga_row = nsga_results.iloc[0]

best_nsga_classifier_name = best_nsga_row['Classifier (Judge)']
best_nsga_gini = float(best_nsga_row['Gini (Real Test)']) # Converting from string if necessary

print(f"Selecting NSGA-II specifically. Best judge found: {best_nsga_classifier_name} (Gini: {best_nsga_gini:.3f})")

# 2. Recovering the features of our winner
winner_features = dictionary_features['NSGA-II']

# Restricting the data exclusively to the features selected
X_train_final = X_train_balanced[winner_features]
X_test_final = X_test_scaled[winner_features]

# 3. Training the ultimate model (Judge winner)
definitive_model = judgment[best_nsga_classifier_name]
print(f"Training the final {best_nsga_classifier_name} model with NSGA-II features...")
definitive_model.fit(X_train_final, y_train_balanced)

# 4. SHAP Intelligence (Ensures it will work independently of the Judge)
print("Calculating SHAP values (Routing to the correct Explainer)...")

if best_nsga_classifier_name in ['LightGBM', 'Random Forest']:
    explainer = shap.TreeExplainer(definitive_model)
    shap_values = explainer.shap_values(X_test_final)

elif best_nsga_classifier_name == 'Logistic Regression':
    explainer = shap.LinearExplainer(definitive_model, X_train_final)
    shap_values = explainer.shap_values(X_test_final)

else:
    print("Initializing KernelExplainer for Neural Network...")
    background_sample = shap.sample(X_train_final, 100)
    explainer = shap.KernelExplainer(definitive_model.predict_proba, background_sample)
    shap_values = explainer.shap_values(X_test_final)

# 5. Compatibility handling
# For binary sorting, LightGBM sometimes returns a list of values ​​[Class 0, Class 1]. We want to explain Class 1 (Target)
if isinstance(shap_values, list):
    shap_values_target = shap_values[1]
elif len(np.array(shap_values).shape) == 3:
    shap_values_target = shap_values[:, :, 1]
else:
    shap_values_target = shap_values

print("Process completed!")

## 5.1: SHAP Summary Plot

In [ ]:
# ==========================================
# Generating the graph
# ==========================================
plt.figure(figsize=(12, 8))

# The summary_plot generates the graph automatically
# max_display=10 focuses on the 10 most important variables to avoid visual clutter.
shap.summary_plot(shap_values_target, X_test_final, max_display=10, show=False)

plt.title(f"SHAP Summary Plot - Top 10 Features Impact\n(NSGA-II + {best_nsga_classifier_name})", 
          fontsize=16, pad=20, fontweight='bold')
plt.xlabel("SHAP value (Impact on Model Output)", fontsize=12)

plt.tight_layout()
plt.show()

## 5.2: Non-Linear Synergy (SHAP Dependency Plot) ---

In [ ]:
# Automatically finding the most important feature
mean_shap_values = np.abs(shap_values_target).mean(axis=0)
top_feature_index = np.argmax(mean_shap_values)
top_feature_name = X_test_final.columns[top_feature_index]

print(f"\nGenerating SHAP Dependency Plot automatically for the Top Feature: '{top_feature_name}'...")

plt.figure(figsize=(10, 6))
shap.dependence_plot(
    top_feature_name, 
    shap_values_target, 
    X_test_final, 
    interaction_index="auto", # Forcing the vital partner
    show=False,
    xmin="percentile(1)",  # Cut the 1% of companies with abnormally low interest rates
    xmax="percentile(99)"  # Cut the 1% of companies with abnormally high interest rates
)
plt.title(f"Hidden Synergy: {top_feature_name} and its vital partner", 
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# --- STEP 6: Validation and Analysis ---

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix

# Calculating the final probabilities for the test set
# (This needs to be done here before plotting the PR curve or doing threshold tuning)
y_pred_proba_final = definitive_model.predict_proba(X_test_final)[:, 1]

## 6.1: Operational Validation (Precision-Recall Curve) ---

In [ ]:
# Calculation of the Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba_final)
ap_score = average_precision_score(y_test, y_pred_proba_final)

# Plot: PR Curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='#2c3e50', lw=3, label=f'AP Score = {ap_score:.4f}')
plt.fill_between(recall, precision, alpha=0.2, color='#34495e')
plt.xlabel('Recall (Ability to Detect the Target))')
plt.ylabel('Precision (Trust in the Alert)')
plt.title('Precision-Recall Curve: GA-Optimized Model', fontsize=14, fontweight='bold')
plt.legend(loc='lower left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print(f"\nAverage Precision (AP): {ap_score:.4f}")

## 6.2: Sensitivity Analysis (Threshold Tuning) ---

In [ ]:
# THRESHOLD TUNING
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
results = []
best_threshold = 0.5 # Default starting point
max_f1 = 0.0

print("\n" + "="*60)
print("IMPACT OF THE DECISION THRESHOLD (TRADE-OFF)")
print("="*60)
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("─" * 60)

for threshold in thresholds:
    # Converts probability to 0 or 1 based on the current cutoff.
    y_bin = (y_pred_proba_final >= threshold).astype(int)
    
    tp = np.sum((y_bin == 1) & (y_test == 1)) # True Positives
    fp = np.sum((y_bin == 1) & (y_test == 0)) # False Positives
    fn = np.sum((y_bin == 0) & (y_test == 1)) # False Negatives
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    # F1 Score
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    results.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

    if f1 > max_f1:
        max_f1 = f1
        best_threshold = threshold

for r in results:
    marker = ""
    if r['threshold'] == 0.5: marker += "(*)"
    if r['threshold'] == best_threshold: marker += " (Best)"
    
    print(f"{r['threshold']:<12.1f} {r['precision']:<12.4f} {r['recall']:<12.4f} {r['f1']:<12.4f} {marker}")

print("\n(*) Standard threshold for classification models.")

## 6.3: Result Analysis (Confusion Matrix) ---

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix

# Confusion Matrix at Threshold of 'best_threshold'
y_pred_final = (y_pred_proba_final >= best_threshold).astype(int)
cm = confusion_matrix(y_test, y_pred_final)

# Plot: Heatmap of the Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False,
            annot_kws={"size": 16, "fontweight": "bold"})

plt.title(f'Confusion Matrix (Threshold = {best_threshold})', fontsize=12)
plt.ylabel('Reality (0: Negative | 1: Positive Target)')
plt.xlabel('Prediction by the Model')
plt.tight_layout()
plt.show()